# PHASE 7: the complete Western Short through the existing hybrid pipeline
Branch `ricks-2d-core`, **T4 GPU**, **Run all**. 460 frames, 720x1280, 30 fps, CPU three.js base (Colab Chrome has no GPU), the existing `lamp_reflect` layer on 201-210, existing QA incl. `check_film.py` on the full video. Expect about **40-50 min** (base about 4.5 s/frame x 460 plus lock 45 s, layer 70 s, QA). No new layers, nothing tuned. One download: `p7_results.zip` (final.mp4, QA, timing, review pack).

In [ ]:
%%capture cap
import os, shutil, subprocess
BRANCH = "ricks-2d-core"; REPO = "https://github.com/damarlanandu2005-jpg/rick"
if not os.path.isdir("/content/rick/.git"):
    r = subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, "/content/rick"], capture_output=True, text=True)
    print(r.stdout[-400:], r.stderr[-400:]); assert r.returncode == 0, "git clone failed"
else:
    print(subprocess.run("cd /content/rick && git pull --ff-only", shell=True, capture_output=True, text=True).stdout)
os.makedirs("/content/hopeall/kaggle_kit", exist_ok=True)   # work folder the setup helpers expect (empty)
print("branch ready: /content/rick")

In [ ]:
%%capture cap
# STEP 0 - BOOTSTRAP: writes ONE shared helper module (all paths, script paths and helpers live there), then imports it.
# Every later cell imports the same constants, so a script is always executed from the exact path it was written to.
from pathlib import Path
import sys

Path("/content/eevee_test/lib").mkdir(parents=True, exist_ok=True)
Path("/content/eevee_test/lib/ee_common.py").write_text(r'''
import os, sys, re, json, time, shutil, hashlib, platform, subprocess, threading, ast
from pathlib import Path

# ---------------- single source of truth for every path ----------------
DATASET = Path("/content/hopeall")
KIT = DATASET / "kaggle_kit"
BLEND = KIT / "gpu_probe.blend"
WORK = Path("/content/eevee_test")
LIB = WORK / "lib"
SCRIPTS = WORK / "scripts"
LOGS = WORK / "logs"
RESULTS = WORK / "results"
PROBE_DIR = WORK / "probe"
BENCH_DIR = WORK / "benchmark"
REPORTS = BENCH_DIR / "reports"
FRAMES_DIR = BENCH_DIR / "frames"
NVGL = WORK / "nvidia_gl"
STATE_FILE = WORK / "state.json"

BLENDER_VERSION = "5.2.4"
BLENDER_DIST = WORK / "blender_dist"
BLENDER_BIN = BLENDER_DIST / ("blender-" + BLENDER_VERSION + "-linux-x64") / "blender"

# generated Blender scripts: each is written with write_script(X, ...) and executed with run_blender(X, ...)
BLENDER_INFO = SCRIPTS / "blender_info.py"
SCENE_INFO = SCRIPTS / "scene_info.py"
GL_PROBE = SCRIPTS / "gl_probe.py"
BENCH = SCRIPTS / "eevee_bench.py"

ALLOW_NON_T4 = False        # set True to allow a different NVIDIA GPU
MAX_BENCH_FRAMES = 60       # hard guard: this notebook never renders a production-length sequence


class CheckpointFailed(RuntimeError):
    pass


def banner(n, title):
    print("=" * 72)
    print("CHECKPOINT %s - %s" % (n, title))
    print("=" * 72)


def passed(n):
    print("\nCHECKPOINT %s PASSED" % n)


def fail(n, msg):
    print("\nCHECKPOINT %s FAILED: %s" % (n, msg))
    raise CheckpointFailed("CHECKPOINT %s: %s" % (n, msg))


def load_state():
    try:
        return json.loads(STATE_FILE.read_text())
    except Exception:
        return {}


def save_state(**kw):
    s = load_state()
    s.update(kw)
    WORK.mkdir(parents=True, exist_ok=True)
    STATE_FILE.write_text(json.dumps(s, indent=2, default=str))
    return s


def sh(cmd, timeout=None):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def nvidia_query():
    r = sh("nvidia-smi --query-gpu=index,name,memory.total,driver_version --format=csv,noheader,nounits")
    if r.returncode != 0:
        return []
    out = []
    for line in r.stdout.strip().splitlines():
        p = [x.strip() for x in line.split(",")]
        if len(p) >= 4:
            out.append({"index": p[0], "name": p[1], "memory_mib": int(float(p[2])), "driver": p[3]})
    return out


def require_file(path):
    ok = subprocess.run(["test", "-f", str(path)]).returncode == 0
    print("[test -f] %s -> %s" % (path, "OK" if ok else "MISSING"))
    if not ok:
        raise FileNotFoundError(str(path))
    return str(path)


def write_script(path, text):
    path = Path(path)
    if path.parent != SCRIPTS:
        raise ValueError("scripts must live directly in %s, got %s" % (SCRIPTS, path))
    if ("eevee_test" + "/eevee_test") in str(path):
        raise ValueError("nested path: %s" % path)
    SCRIPTS.mkdir(parents=True, exist_ok=True)
    ast.parse(text)  # syntax check before it ever reaches Blender
    path.write_text(text)
    require_file(path)
    print("[script written] %s (%d bytes, syntax OK)" % (path, len(text)))
    return path


def gl_env(backend=None):
    env = os.environ.copy()
    st = load_state()
    if st.get("gl_lib_dir"):
        old = env.get("LD_LIBRARY_PATH", "")
        env["LD_LIBRARY_PATH"] = st["gl_lib_dir"] + (":" + old if old else "")
    if st.get("egl_vendor_json"):
        env["__EGL_VENDOR_LIBRARY_FILENAMES"] = st["egl_vendor_json"]
    if backend == "vulkan" and st.get("vk_icd_json"):
        env["VK_ICD_FILENAMES"] = st["vk_icd_json"]
    env.pop("LIBGL_ALWAYS_SOFTWARE", None)
    return env


def run_blender(script, script_args=(), log_path=None, timeout=1800, backend=None, show=None, hide=r"EGL Error"):
    """Run Blender headless on `script`. Verifies binary and script with `test -f` first. Full log is saved;
    lines matching `hide` (EGL probe noise) are counted, not printed, never discarded."""
    require_file(BLENDER_BIN)
    if not os.access(BLENDER_BIN, os.X_OK):
        raise PermissionError("not executable: %s" % BLENDER_BIN)
    script = Path(script)
    require_file(script)
    log_path = Path(log_path) if log_path else LOGS / (script.stem + ".log")
    log_path.parent.mkdir(parents=True, exist_ok=True)
    cmd = [str(BLENDER_BIN), "-b", "--factory-startup", "--python-exit-code", "1"]
    if backend:
        cmd += ["--gpu-backend", backend]
    cmd += ["-P", str(script), "--"] + [str(x) for x in script_args]
    print("[run]", " ".join(cmd))
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            errors="replace", bufsize=1, env=gl_env(backend))
    flag = {"timeout": False}

    def _kill():
        flag["timeout"] = True
        proc.kill()

    timer = threading.Timer(timeout, _kill)
    timer.start()
    lines, hidden = [], 0
    with open(log_path, "w") as lf:
        for line in proc.stdout:
            lf.write(line)
            lines.append(line.rstrip("\n"))
            if hide and re.search(hide, line):
                hidden += 1
                continue
            if show and re.search(show, line):
                print(line.rstrip())
    rc = proc.wait()
    timer.cancel()
    return {"rc": rc, "lines": lines, "hidden_noise": hidden, "timed_out": flag["timeout"],
            "log": str(log_path), "cmd": cmd}


def parse_tagged(lines, tag):
    out = None
    for l in lines:
        if l.startswith(tag + " "):
            try:
                out = json.loads(l[len(tag) + 1:])
            except Exception:
                pass
    return out


ERR_RE = re.compile(r"Traceback \(most recent|^Error:|Segmentation fault|Fatal|CUDA error|shader compilation failed|"
                    r"Failed to compile|out of memory|Unable to open a display", re.I)


def scan_errors(lines):
    errs = [l for l in lines if ERR_RE.search(l) and "EGL Error" not in l]
    egl = [l for l in lines if "EGL Error" in l]
    return errs, egl


def start_smi(csv_path):
    f = open(csv_path, "w")
    p = subprocess.Popen(["nvidia-smi", "--query-gpu=timestamp,index,utilization.gpu,memory.used",
                          "--format=csv,noheader,nounits", "-lms", "500"], stdout=f, stderr=subprocess.DEVNULL)
    return p, f


def stop_smi(p, f):
    p.terminate()
    try:
        p.wait(5)
    except Exception:
        p.kill()
    f.close()


def smi_peaks(csv_path):
    peak = {"util_pct": 0.0, "mem_mib": 0.0, "samples": 0}
    try:
        for l in open(csv_path):
            x = [t.strip() for t in l.split(",")]
            if len(x) < 4:
                continue
            try:
                peak["util_pct"] = max(peak["util_pct"], float(x[2]))
                peak["mem_mib"] = max(peak["mem_mib"], float(x[3]))
                peak["samples"] += 1
            except ValueError:
                pass
    except FileNotFoundError:
        pass
    return peak


# ---------------- code prepended to every generated Blender script ----------------
BPY_COMMON = r"""
import bpy, sys, os, json, time, traceback

def script_args():
    return sys.argv[sys.argv.index("--") + 1:] if "--" in sys.argv else []

def available_engines():
    return [e.identifier for e in bpy.types.RenderSettings.bl_rna.properties["engine"].enum_items]

def pick_eevee(engines):
    cands = [e for e in engines if "EEVEE" in e.upper()]
    if not cands:
        raise RuntimeError("No EEVEE engine in this Blender build. Available: %s" % engines)
    nxt = [e for e in cands if "NEXT" in e.upper()]
    return (nxt or sorted(cands))[0]

def gl_info():
    import gpu
    p = gpu.platform
    d = {"vendor": p.vendor_get(), "renderer": p.renderer_get(), "version": p.version_get()}
    for key, fn in (("backend", "backend_type_get"), ("device_type", "device_type_get")):
        try:
            d[key] = getattr(p, fn)()
        except Exception as e:
            d[key] = "unavailable: %r" % (e,)
    return d

def classify_gl(d):
    text = (str(d.get("vendor")) + " " + str(d.get("renderer"))).lower()
    soft = any(k in text for k in ("llvmpipe", "softpipe", "swrast", "lavapipe", "software", "mesa"))
    if soft or str(d.get("device_type")).upper() == "SOFTWARE":
        return "SOFTWARE"
    if "nvidia" in text:
        return "NVIDIA_GPU"
    return "OTHER_GPU"

def file_ok(p):
    return os.path.isfile(p) and os.path.getsize(p) > 0
"""
''')

sys.path.insert(0, "/content/eevee_test/lib")
import importlib, ee_common
importlib.reload(ee_common)
from ee_common import *

print("helper module:", ee_common.__file__)
print("WORK    :", WORK)
print("KIT     :", KIT)
print("BLEND   :", BLEND)
print("BLENDER :", BLENDER_BIN)
print("SCRIPTS :", [str(p) for p in (BLENDER_INFO, SCENE_INFO, GL_PROBE, BENCH)])


In [ ]:
%%capture cap
# STEP 0b - nothing to restore. The old casino dataset is no longer used by this test.
print('no dataset needed: the street is built by fulltest/street_kit.py inside Blender; the GPU probe scene is generated below.')

In [ ]:
%%capture cap
# CHECKPOINT 0 - Environment
import sys
sys.path.insert(0, "/content/eevee_test/lib")
from ee_common import *

banner(0, "Environment")
print("Python:", platform.python_version(), "|", platform.platform())
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print("Google Colab:", IN_COLAB)
if not IN_COLAB:
    print("WARNING: google.colab not importable - this notebook is meant for Colab.")

gpus = nvidia_query()
if not gpus:
    fail(0, "nvidia-smi sees no GPU. Runtime > Change runtime type > T4 GPU.")
for g in gpus:
    print("GPU %(index)s: %(name)s | %(memory_mib)d MiB | driver %(driver)s" % g)
if not any("T4" in g["name"] for g in gpus) and not ALLOW_NON_T4:
    fail(0, "GPU is not a Tesla T4 (%s). Set ALLOW_NON_T4 = True in ee_common to override." % gpus[0]["name"])
print("NVIDIA_DRIVER_CAPABILITIES =", os.environ.get("NVIDIA_DRIVER_CAPABILITIES"))

mem_kb = int(re.search(r"MemTotal:\s+(\d+)", open("/proc/meminfo").read()).group(1))
print("CPU cores: %s | RAM: %.1f GB" % (os.cpu_count(), mem_kb / 1048576))
du = shutil.disk_usage("/content")
print("Disk /content: %.1f GB free of %.1f GB" % (du.free / 1e9, du.total / 1e9))
if du.free < 8e9:
    fail(0, "less than 8 GB free disk (Blender ~1.2 GB + NVIDIA GL userspace libs ~1 GB temporary).")

if not DATASET.is_dir():
    fail(0, "dataset directory missing: %s - run the Step 0b cell above." % DATASET)
if not KIT.is_dir():
    fail(0, "kit directory missing: %s" % KIT)
print("Kit files:")
for p in sorted(KIT.iterdir()):
    print("  %-48s %10d bytes" % (p.name, p.stat().st_size))
blend_hash = "generated-later"

# Informational only: the kit's own helper files are NOT executed by this notebook.
SEARCH = "/" + "kag" + "gle" + "/"
for p in sorted(KIT.glob("*")):
    if p.suffix in (".py", ".sh", ".ipynb"):
        try:
            n = p.read_text(errors="replace").count(SEARCH)
        except Exception:
            continue
        if n:
            print("note: kit file %s has %d Kaggle-specific path refs (not used by this notebook)" % (p.name, n))

for d in (SCRIPTS, LOGS, RESULTS, PROBE_DIR, REPORTS, FRAMES_DIR, BLENDER_DIST, NVGL):
    d.mkdir(parents=True, exist_ok=True)
save_state(blend_sha256=blend_hash, gpu=gpus[0], in_colab=IN_COLAB)
passed(0)


In [ ]:
%%capture cap
# CHECKPOINT 1 - Blender 5.2.x
import sys, urllib.request
sys.path.insert(0, "/content/eevee_test/lib")
from ee_common import *

banner(1, "Blender")
V = BLENDER_VERSION
MM = ".".join(V.split(".")[:2])

# Runtime libraries Blender needs on a bare Colab image (no NVIDIA driver is installed here).
need = ["libegl1", "libgl1", "libxi6", "libxxf86vm1", "libxfixes3", "libxrender1",
        "libxkbcommon0", "libsm6", "libxext6", "libvulkan1"]
r = sh("apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq " + " ".join(need), timeout=900)
print("apt runtime deps exit code:", r.returncode)
if r.returncode != 0:
    print((r.stderr or r.stdout)[-1500:])

if BLENDER_BIN.is_file() and os.access(BLENDER_BIN, os.X_OK):
    print("Blender already installed at", BLENDER_BIN)
else:
    base = "https://download.blender.org/release/Blender%s" % MM
    tarball = BLENDER_DIST / "blender.tar.xz"
    BLENDER_DIST.mkdir(parents=True, exist_ok=True)
    url = "%s/blender-%s-linux-x64.tar.xz" % (base, V)
    print("downloading", url)
    r = sh("curl -fL --retry 3 -sS -o '%s' '%s'" % (tarball, url), timeout=1800)
    if r.returncode != 0:
        fail(1, "download failed (is internet on?): %s" % r.stderr.strip()[-300:])
    print("downloaded %.0f MB" % (tarball.stat().st_size / 1e6))
    verified = None
    try:
        fetch = lambda u: urllib.request.urlopen(u, timeout=60).read().decode()
        listing = fetch(base + "/")
        m = re.search(r"blender-%s[^\"<>\s]*\.sha256" % re.escape(V), listing)
        sums = fetch(base + "/" + m.group(0))
        expected = [l.split()[0] for l in sums.splitlines() if "linux-x64.tar.xz" in l][0]
        verified = (sha256_file(tarball) == expected)
    except Exception as e:
        print("WARNING: could not fetch checksum (%r) - tarball unverified" % (e,))
    if verified is False:
        fail(1, "SHA256 mismatch for the Blender download")
    print("sha256 verified:", verified)
    r = sh("tar -xf '%s' -C '%s'" % (tarball, BLENDER_DIST), timeout=900)
    if r.returncode != 0:
        fail(1, "tar extraction failed: %s" % r.stderr[-300:])
    tarball.unlink()

require_file(BLENDER_BIN)
if not os.access(BLENDER_BIN, os.X_OK):
    fail(1, "Blender binary is not executable")

missing = sh("ldd '%s' | grep 'not found'" % BLENDER_BIN).stdout.strip()
if missing:
    fail(1, "Blender has unresolved shared libraries:\n" + missing)
print("ldd: all Blender shared libraries resolved")

r = sh("timeout 120 '%s' --version" % BLENDER_BIN)
print(r.stdout.strip())
m = re.search(r"Blender (\d+)\.(\d+)\.(\d+)", r.stdout)
if not m:
    fail(1, "could not parse `blender --version`: %r" % r.stdout[:200])
found = "%s.%s.%s" % m.groups()
if (m.group(1), m.group(2)) != tuple(MM.split(".")):
    fail(1, "wrong Blender version %s (need %s.x)" % (found, MM))
if found != V:
    print("WARNING: expected %s but found %s (still %s.x)" % (V, found, MM))

write_script(BLENDER_INFO, BPY_COMMON + r'''
def s(x):
    return x.decode() if isinstance(x, bytes) else str(x)
engines = available_engines()
info = {"version_string": bpy.app.version_string, "version": list(bpy.app.version),
        "version_cycle": s(bpy.app.version_cycle), "build_hash": s(bpy.app.build_hash),
        "build_date": s(bpy.app.build_date), "build_time": s(bpy.app.build_time),
        "build_branch": s(bpy.app.build_branch), "python": sys.version.split()[0],
        "binary_path": bpy.app.binary_path, "engines": engines}
print("AVAILABLE_ENGINES", engines)
print("BLENDER_INFO_JSON", json.dumps(info, default=str))
''')
res = run_blender(BLENDER_INFO, [], log_path=LOGS / "blender_info.log", timeout=300,
                  show=r"AVAILABLE_ENGINES|BLENDER_INFO_JSON|Traceback|Error")
info = parse_tagged(res["lines"], "BLENDER_INFO_JSON")
if res["rc"] != 0 or not info:
    fail(1, "blender_info.py failed (rc=%s). See %s" % (res["rc"], res["log"]))
print("Blender version :", info["version_string"])
print("Blender build   : hash %s | %s %s | branch %s" % (info["build_hash"], info["build_date"], info["build_time"], info["build_branch"]))
print("Blender Python  :", info["python"])
if tuple(info["version"][:2]) != tuple(int(x) for x in MM.split(".")):
    fail(1, "bpy reports version %s, need %s.x" % (info["version"], MM))
save_state(blender=info)
passed(1)


In [ ]:
%%capture cap
# CHECKPOINT 2 - Scene inspection (read-only)
import sys
sys.path.insert(0, "/content/eevee_test/lib")
from ee_common import *

banner(2, "Scene")
SCENE_MAKE = SCRIPTS / "make_probe.py"
write_script(SCENE_MAKE, BPY_COMMON + r'''
a = script_args()
bpy.ops.wm.read_factory_settings(use_empty=True)
s = bpy.context.scene
pm = bpy.data.meshes.new("plane"); pm.from_pydata([(-2,-2,0),(2,-2,0),(2,2,0),(-2,2,0)], [], [(0,1,2,3)]); pm.update()
s.collection.objects.link(bpy.data.objects.new("plane", pm))
cam = bpy.data.objects.new("cam", bpy.data.cameras.new("cam")); s.collection.objects.link(cam); cam.location = (0, -6, 2); cam.rotation_euler = (1.25, 0, 0); s.camera = cam
sun = bpy.data.objects.new("sun", bpy.data.lights.new("sun", "SUN")); s.collection.objects.link(sun); sun.rotation_euler = (0.8, 0.2, 0.5)
s.frame_start = 1; s.frame_end = 1
bpy.ops.wm.save_as_mainfile(filepath=a[0])
print("PROBE_SCENE_SAVED", a[0])
''')
res = run_blender(SCENE_MAKE, [BLEND], log_path=LOGS / "make_probe.log", timeout=300, show=r"PROBE_SCENE_SAVED|Traceback|Error")
if res["rc"] != 0: fail(2, "could not create the tiny GPU probe scene, see %s" % res["log"])
save_state(blend_sha256=sha256_file(BLEND))
require_file(BLEND)
write_script(SCENE_INFO, BPY_COMMON + r'''
a = script_args()
bpy.ops.wm.open_mainfile(filepath=a[0], load_ui=False)
engines = available_engines()
s = bpy.context.scene
missing = [i.name for i in bpy.data.images
           if i.source == "FILE" and not i.packed_file and not os.path.exists(bpy.path.abspath(i.filepath))]
info = {
    "scene_name": s.name, "all_scenes": [x.name for x in bpy.data.scenes],
    "objects_in_scene": len(s.objects), "objects_total": len(bpy.data.objects),
    "mesh_datablocks": len(bpy.data.meshes),
    "mesh_objects": sum(1 for o in bpy.data.objects if o.type == "MESH"),
    "lights": sum(1 for o in bpy.data.objects if o.type == "LIGHT"),
    "materials": len(bpy.data.materials),
    "camera": s.camera.name if s.camera else None,
    "frame_start": s.frame_start, "frame_end": s.frame_end,
    "fps": s.render.fps, "fps_base": s.render.fps_base,
    "res_x": s.render.resolution_x, "res_y": s.render.resolution_y, "res_pct": s.render.resolution_percentage,
    "engine_in_file": s.render.engine, "available_engines": engines, "eevee_engine_that_will_be_used": pick_eevee(engines),
    "file_format": s.render.image_settings.file_format, "missing_external_images": missing,
}
print("AVAILABLE_ENGINES", engines)
print("SCENE_INFO_JSON", json.dumps(info, default=str))
''')
res = run_blender(SCENE_INFO, [BLEND], log_path=LOGS / "scene_info.log", timeout=600,
                  show=r"AVAILABLE_ENGINES|Traceback|Error")
sc = parse_tagged(res["lines"], "SCENE_INFO_JSON")
if res["rc"] != 0 or not sc:
    fail(2, "scene_info.py failed (rc=%s). See %s" % (res["rc"], res["log"]))

for k in ("scene_name", "objects_in_scene", "mesh_objects", "mesh_datablocks", "materials", "camera",
          "frame_start", "frame_end", "res_x", "res_y", "res_pct", "engine_in_file", "eevee_engine_that_will_be_used"):
    print("%-32s %s" % (k, sc[k]))
print("%-32s %s (fps %s / base %s)" % ("FPS", sc["fps"] / sc["fps_base"], sc["fps"], sc["fps_base"]))
if sc["missing_external_images"]:
    print("WARNING: missing external image files (would render pink):", sc["missing_external_images"][:10])
if not sc["camera"]:
    fail(2, "scene has no active camera")
if sc["objects_in_scene"] == 0:
    fail(2, "scene has no objects")
if sc["frame_end"] < sc["frame_start"]:
    fail(2, "invalid frame range")
if sha256_file(BLEND) != load_state()["blend_sha256"]:
    fail(2, "ORIGINAL .blend CHANGED during inspection")
print("original .blend unchanged (sha256 verified)")
save_state(scene=sc)
passed(2)


In [ ]:
%%capture cap
# CHECKPOINT 3 (part A) - locate / provision the NVIDIA EGL userspace libraries
import sys
sys.path.insert(0, "/content/eevee_test/lib")
from ee_common import *

banner(3, "GPU / GL diagnosis - part A: NVIDIA EGL userspace libraries")
# Colab exposes nvidia-smi / CUDA but often NOT the graphics (EGL/GLX) libraries, so Blender cannot reach the T4 via GL.
# If they are missing we extract EXACTLY the libraries matching the running kernel driver from NVIDIA's installer.
# This is --extract-only into a private folder: nothing is installed system-wide and the driver is not touched.
PROVISION_NVIDIA_GL_USERSPACE = True

gpus = nvidia_query()
if not gpus:
    fail(3, "nvidia-smi sees no GPU")
drv = gpus[0]["driver"]
print("kernel driver version:", drv)

roots = [d for d in ["/usr/lib", "/usr/lib64", "/lib", "/usr/local/nvidia", "/usr/local/cuda", "/opt",
                     "/usr/share/glvnd", "/usr/share/vulkan", "/etc/vulkan", "/etc/glvnd"] if os.path.isdir(d)]
found = {}
for pat in ("libEGL_nvidia.so*", "libGLX_nvidia.so*", "libnvidia-glcore.so*", "nvidia_icd.json", "10_nvidia.json"):
    r = sh("find %s -name '%s' 2>/dev/null | head -20" % (" ".join(roots), pat))
    found[pat] = [x for x in r.stdout.split() if x]
    print("%-24s %s" % (pat, found[pat]))
print("ldconfig NVIDIA/EGL entries:\n" + (sh("ldconfig -p | grep -i -E 'nvidia|libEGL|libGLX' | head -20").stdout or "  (none)\n"))

matches = lambda p: os.path.realpath(p).endswith("." + drv)
system_egl = [p for p in found["libEGL_nvidia.so*"] if matches(p)]
lib_dir, source = None, None

if system_egl:
    lib_dir, source = os.path.dirname(system_egl[0]), "system"
    print("Found system NVIDIA EGL matching driver %s: %s" % (drv, system_egl[0]))
else:
    stale = found["libEGL_nvidia.so*"]
    print("No libEGL_nvidia matching driver %s on this system%s." % (drv, (" (version-mismatched copies: %s)" % stale) if stale else ""))
    if not PROVISION_NVIDIA_GL_USERSPACE:
        fail(3, "NVIDIA EGL libraries unavailable and PROVISION_NVIDIA_GL_USERSPACE is False.")
    run_file = NVGL / ("NVIDIA-Linux-x86_64-%s.run" % drv)
    extract = NVGL / "extract"
    libdir = NVGL / "lib"
    urls = ["https://us.download.nvidia.com/tesla/%s/NVIDIA-Linux-x86_64-%s.run" % (drv, drv),
            "https://us.download.nvidia.com/XFree86/Linux-x86_64/%s/NVIDIA-Linux-x86_64-%s.run" % (drv, drv)]
    ok = run_file.is_file() and run_file.stat().st_size > 100e6
    for u in ([] if ok else urls):
        print("downloading", u)
        r = sh("curl -fL --retry 2 -sS -o '%s' '%s'" % (run_file, u), timeout=1800)
        if r.returncode == 0 and run_file.is_file() and run_file.stat().st_size > 100e6:
            ok = True
            break
        print("  failed:", r.stderr.strip()[-200:])
    if not ok:
        fail(3, "could not download the NVIDIA %s userspace package; Blender cannot reach the T4 through GL without it." % drv)
    if extract.exists():
        shutil.rmtree(extract)
    r = sh("sh '%s' --extract-only --target '%s'" % (run_file, extract), timeout=900)
    if r.returncode != 0:
        fail(3, "extract-only failed: %s" % (r.stderr or r.stdout)[-300:])
    if not list(extract.glob("libEGL_nvidia.so.*")):
        fail(3, "the NVIDIA package for %s ships no libEGL_nvidia - GL cannot use this GPU here." % drv)
    if libdir.exists():
        shutil.rmtree(libdir)
    libdir.mkdir(parents=True)
    keep = ("libEGL_nvidia.so", "libGLX_nvidia.so", "libGLESv1_CM_nvidia.so", "libGLESv2_nvidia.so",
            "libnvidia-eglcore.so", "libnvidia-glcore.so", "libnvidia-glsi.so", "libnvidia-tls.so",
            "libnvidia-glvkspirv.so", "libnvidia-gpucomp.so", "libnvidia-rtcore.so")
    for f in extract.iterdir():
        if f.name.startswith(keep) and f.is_file():
            shutil.copy2(f, libdir / f.name)
    for base, so in (("libEGL_nvidia", "0"), ("libGLX_nvidia", "0"), ("libGLESv2_nvidia", "2"), ("libGLESv1_CM_nvidia", "1")):
        tgt = libdir / ("%s.so.%s" % (base, drv))
        lnk = libdir / ("%s.so.%s" % (base, so))
        if tgt.exists() and not lnk.exists():
            lnk.symlink_to(tgt.name)
    if (extract / "nvidia_icd.json").is_file():
        shutil.copy2(extract / "nvidia_icd.json", libdir / "nvidia_icd.json")
    shutil.rmtree(extract, ignore_errors=True)
    run_file.unlink()
    lib_dir, source = str(libdir), "extracted-from-nvidia-installer"
    print("Copied matching GL userspace libraries to", lib_dir)

egl_lib = Path(lib_dir) / "libEGL_nvidia.so.0"
if not egl_lib.exists():
    cands = sorted(Path(lib_dir).glob("libEGL_nvidia.so.*"))
    if not cands:
        fail(3, "libEGL_nvidia not found in %s" % lib_dir)
    egl_lib = cands[0]
unres = sh("LD_LIBRARY_PATH='%s' ldd '%s' | grep 'not found'" % (lib_dir, egl_lib)).stdout.strip()
if unres:
    print("WARNING: libEGL_nvidia has unresolved dependencies:\n" + unres)

# This JSON points at the REAL library found/extracted above (not a fabricated config).
egl_json = WORK / "egl_nvidia.json"
egl_json.write_text(json.dumps({"file_format_version": "1.0.0", "ICD": {"library_path": str(egl_lib)}}))
vk_icd = Path(lib_dir) / "nvidia_icd.json"
save_state(gl_lib_dir=str(lib_dir), egl_vendor_json=str(egl_json), gl_source=source,
           vk_icd_json=str(vk_icd) if vk_icd.is_file() else None, nvidia_egl_lib=str(egl_lib))
print("GL source: %s | EGL lib: %s | vendor json: %s" % (source, egl_lib, egl_json))
print("(setup step only - CHECKPOINT 3 is decided by the Blender probe in the next cell)")


In [ ]:
%%capture cap
# CHECKPOINT 3 (part B) - GPU / EEVEE smoke test via Blender itself
import sys
sys.path.insert(0, "/content/eevee_test/lib")
from ee_common import *

banner(3, "GPU / EEVEE smoke test (64x64)")
require_file(BLEND)
write_script(GL_PROBE, BPY_COMMON + r'''
a = script_args()
blend, out_png, report = a[0], a[1], a[2]
def main():
    engines = available_engines()
    print("AVAILABLE_ENGINES", engines)
    bpy.ops.wm.open_mainfile(filepath=blend, load_ui=False)
    s = bpy.context.scene
    eng = pick_eevee(engines)
    print("SELECTED_EEVEE_ENGINE", eng)
    s.render.engine = eng
    s.render.resolution_x = 64
    s.render.resolution_y = 64
    s.render.resolution_percentage = 100
    if hasattr(s.eevee, "taa_render_samples"):
        s.eevee.taa_render_samples = 4
    s.render.image_settings.file_format = "PNG"
    s.render.use_file_extension = False
    s.render.filepath = out_png
    s.frame_set(s.frame_start)
    os.makedirs(os.path.dirname(out_png), exist_ok=True)
    t = time.perf_counter()
    bpy.ops.render.render(write_still=True)
    dt = time.perf_counter() - t
    info = gl_info()
    verdict = classify_gl(info)
    rep = {"blender": bpy.app.version_string, "available_engines": engines, "selected_eevee_engine": eng,
           "gl": info, "gl_verdict": verdict, "render_s": round(dt, 3), "output_ok": file_ok(out_png), "output": out_png}
    with open(report, "w") as f:
        json.dump(rep, f, indent=2, default=str)
    print("GL_INFO", json.dumps(info))
    print("GL_VERDICT", verdict)
    if verdict != "NVIDIA_GPU":
        print("NOT AN NVIDIA GPU RENDERER:", verdict)
        sys.exit(2)
    if not rep["output_ok"]:
        print("probe render produced no output file")
        sys.exit(3)
try:
    main()
except SystemExit:
    raise
except Exception:
    traceback.print_exc()
    sys.exit(1)
''')

chosen, attempts = None, []
for backend in ("opengl", "vulkan"):
    rep_path = PROBE_DIR / ("gl_probe_%s.json" % backend)
    png_path = PROBE_DIR / ("probe_%s.png" % backend)
    for p in (rep_path, png_path):
        if p.exists():
            p.unlink()
    print("\n--- trying GPU backend: %s ---" % backend)
    res = run_blender(GL_PROBE, [BLEND, png_path, rep_path], log_path=LOGS / ("gl_probe_%s.log" % backend),
                      timeout=300, backend=backend,
                      show=r"AVAILABLE_ENGINES|SELECTED_EEVEE_ENGINE|GL_INFO|GL_VERDICT|Traceback|Error|NOT AN NVIDIA")
    rep = json.loads(rep_path.read_text()) if rep_path.is_file() else None
    ok = bool(res["rc"] == 0 and rep and rep["gl_verdict"] == "NVIDIA_GPU" and rep["output_ok"] and png_path.is_file())
    attempts.append({"backend": backend, "rc": res["rc"], "verdict": rep["gl_verdict"] if rep else None,
                     "log": res["log"], "egl_noise_lines": res["hidden_noise"]})
    print("backend %s -> rc=%s verdict=%s EGL-noise-lines=%s" % (backend, res["rc"], rep["gl_verdict"] if rep else "no report", res["hidden_noise"]))
    if ok:
        chosen = (backend, rep)
        break

if not chosen:
    print("\nAttempts:", json.dumps(attempts, indent=2))
    for a in attempts:
        print("\n--- last 25 lines of %s ---" % a["log"])
        print("\n".join(Path(a["log"]).read_text(errors="replace").splitlines()[-25:]))
    reason = "Blender could not render EEVEE on the NVIDIA T4 with the OpenGL or Vulkan backend."
    if any(a["verdict"] == "SOFTWARE" for a in attempts):
        reason = "Blender fell back to SOFTWARE rendering (llvmpipe/Mesa) - this is not a GPU result."
    fail(3, reason + " See logs above. Benchmark stopped on purpose.")

backend, rep = chosen
st = load_state()
print("\n================ GPU / EEVEE SUMMARY ================")
print("Blender version : ", rep["blender"])
print("GPU (nvidia-smi): ", st["gpu"]["name"], "|", st["gpu"]["memory_mib"], "MiB | driver", st["gpu"]["driver"])
print("GL vendor       : ", rep["gl"]["vendor"])
print("GL renderer     : ", rep["gl"]["renderer"])
print("GL version      : ", rep["gl"]["version"])
print("GPU backend     : ", backend, "(Blender reports %s)" % rep["gl"].get("backend"))
print("AVAILABLE_ENGINES", rep["available_engines"])
print("EEVEE engine    : ", rep["selected_eevee_engine"])
print("Rendering on    : ", "NVIDIA GPU (hardware)" if rep["gl_verdict"] == "NVIDIA_GPU" else "NOT NVIDIA")
print("GL libs source  : ", st.get("gl_source"))
save_state(gl_backend=backend, gl_probe=rep)
passed(3)


In [ ]:
%%capture cap
import subprocess, os
def live(cmd, cwd=None):
    p = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True, errors="replace"); return p.returncode, (p.stdout + p.stderr)[-800:]
if subprocess.run("node -e 'process.exit(parseInt(process.versions.node)>=18?0:1)'", shell=True).returncode != 0:
    rc, o = live("curl -fsSL https://deb.nodesource.com/setup_22.x | bash - && apt-get install -y -qq nodejs"); assert rc == 0, o
os.makedirs("/content/pw", exist_ok=True)
rc, o = live("npm install --no-audit --no-fund playwright && npx playwright install --with-deps chromium", cwd="/content/pw"); assert rc == 0, o
rc, o = live("ffmpeg -version | head -1"); print(o)


In [ ]:
# PHASE 7 RUN (the only cell with output)
import sys, os, subprocess, json, zipfile, glob
sys.path.insert(0, "/content/eevee_test/lib")
from ee_common import *
st = load_state(); assert st.get("gl_backend"), "GPU probe (checkpoint 3) did not pass; re-run the cells above"
env = gl_env(st["gl_backend"]); env["PLAYWRIGHT_DIR"] = "/content/pw/node_modules/playwright"
cmd = [sys.executable, "/content/rick/phase7/run_full.py", "--blender", str(BLENDER_BIN), "--gpu-backend", st["gl_backend"], "--out", "/content/p7_out", "--renderer", "cpu"]
p = subprocess.run(cmd, env=env, capture_output=True, text=True, errors="replace")
print(p.stdout[-6000:]); print(p.stderr[-2000:] if p.returncode else "exit 0")
z = "/content/p7_results.zip"; O = "/content/p7_out"
with zipfile.ZipFile(z, "w", zipfile.ZIP_DEFLATED) as zf:
    for q in glob.glob(O + "/review/**", recursive=True) + glob.glob(O + "/*.json") + glob.glob(O + "/*.log") + glob.glob(O + "/ep/out/pipeline/*") + glob.glob(O + "/ep/out/qa/**", recursive=True) + glob.glob(O + "/ep/out/layers/lamp_reflect/*.json") + glob.glob(O + "/ep/out/layers/lamp_reflect/*.log"):
        if os.path.isfile(q): zf.write(q, os.path.relpath(q, O))
print("zip", round(os.path.getsize(z) / 1e6, 1), "MB")
from google.colab import files; files.download(z)
